In [1]:
!pip install jinja2 starlette

In [2]:
import gradio as gr 
from langchain_groq import ChatGroq
import json 
from typing import List, Tuple, Dict, Any 
from langchain_core.messages import HumanMessage, SystemMessage
import os 
import getpass 

In [3]:
def set_if_undefined(var:str): 
    if os.environ.get(var): 
        return 
    os.environ[var] = getpass.getpass(var) 
set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [4]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile", 
    temperature = 0.7, 
    max_tokens = 1024
)
test_query = "Tell me a friendly joke about seattle."
test_response = llm.invoke(test_query)
print(test_response.content)

Why did the coffee file a police report in Seattle?

Because it got mugged.


Build a basic chatbot
Start with a simple chatbot that echoes user messages. This will help you understand Gradio's chat components.

Note: You may see a warning notification indicating that a newer version of Gradio is available. You can safely ignore this warning and proceed with the lab.

In [6]:
def echo_chatbot(message: str, history: List[Tuple[str,str]])-> str:  
    """A simple echo chatbot for testing."""
    return f"You said {message}."
#Create an interface 
demo_echo = gr.ChatInterface(
    fn = echo_chatbot, 
    title = "Echo Chatbot",
    description = "A simple chatbot that echo's your message", 
    examples = [
        "Hi", 
        "Hello."
    ]
    
)
print("Basic chatbot interface created!")
print("To launch, run: demo_echo.launch()")

Basic chatbot interface created!
To launch, run: demo_echo.launch()


Implement intent classification
Now, add logic to classify user intent: Do they want restaurant recommendations, recipe recommendations, or both?

In [7]:
def classify_intent(user_message:str, llm:ChatGroq) -> str :  
    """Classifies user's intent as restaurant,recipe ,both or clarification"""
    system_prompt = """You are an intent classfier for a food recommendation system.
    Analyze the user's message and classify it as ONE of:
- "restaurant" - User wants restaurant recommendations
- "recipe" - User wants recipe recommendations
- "both" - User wants both restaurant and recipe recommendations
- "clarification" - User needs help or is asking a question
- "database" - User wants to add/edit/delete database entries

Examples:
"Where should I eat tonight?" → restaurant
"How do I make lasagna?" → recipe
"I want dinner ideas" → both
"What can you help me with?" → clarification
"I want to add a new restaurant" → database

Respond with ONLY the classification label."""
    messages = [
        SystemMessage(content = system_prompt), 
        HumanMessage(content = user_message)
    ]
    response = llm.invoke(messages)
    #Validate intent
    intent = response.content.lower().strip() 
    if intent not in ["restaurant", "recipe", "both", "clarification", "database"]: 
        intent = "clarification"
    return intent
print("Intent function created")    

Intent function created


In [9]:
# Test intent classification
test_messages = [
    "I'm looking for Italian restaurants",
    "How do I make pad thai?",
    "Give me dinner ideas",
    "What can you do?"
]

print("Testing intent classification:\n")
try:
    for msg in test_messages: 
        intent = classify_intent(msg, llm) 
        print(f"The message is : {msg}")
        print(f"\nThe intent is :{intent}")
except Exception as e: 
    print(f"No valid Api key :{e}")

Testing intent classification:

The message is : I'm looking for Italian restaurants

The intent is :restaurant
The message is : How do I make pad thai?

The intent is :recipe
The message is : Give me dinner ideas

The intent is :both
The message is : What can you do?

The intent is :clarification


Extract user preferences
Once you know the user's intent, you need to extract their preferences from natural language.



In [10]:
def extract_preferences(user_message:str, llm:ChatGroq) -> str : 
    """Extracts the user preference from the natural language"""
    system_prompt = """ You are a preference extractor for a food recommendation system.
    Extract user preferences from their message and return JSON with these keys:
- favorite_cuisines: List of mentioned cuisines (e.g., ["Italian", "Thai"])
- dietary_restrictions: List of dietary needs (e.g., ["vegetarian", "gluten-free"])
- dining_occasion: Type of dining (e.g., "casual", "fine dining", "quick bite")
- price_range: Price preference (e.g., "$", "$$", "$$$", "$$$$")
- flavor_preferences: List of flavor preferences (e.g., ["spicy", "sweet"])
- other_preferences: Any other relevant details

If a field is not mentioned, use an empty list or "not specified".

Example:
Input: "I love spicy Thai food and I'm vegetarian"
Output: {
  "favorite_cuisines": ["Thai"],
  "dietary_restrictions": ["vegetarian"],
  "dining_occasion": "not specified",
  "price_range": "not specified",
  "flavor_preferences": ["spicy"],
  "other_preferences": ""
}

Respond with ONLY valid JSON.
    
    """
    messages = [
        SystemMessage(content = system_prompt), 
        HumanMessage(content = user_message)
    ]
    response = llm.invoke(messages)
    try:
        preferences = json.loads(response.content)
    except: 
         # Fallback if parsing fails
        preferences = {
            "favorite_cuisines": [],
            "dietary_restrictions": [],
            "dining_occasion": "not specified",
            "price_range": "not specified",
            "flavor_preferences": [],
            "other_preferences": ""
        }
    return preferences
print("Extract preference function successfully created!!!")    
        

Extract preference function successfully created!!!


Task 1: Test the preference extraction function
Test the extract_preferences function with the message: "I'm looking for affordable vegetarian Mexican restaurants"

In [11]:
## Type your answer here

test_message = """I'm looking for a affordable vegetarian Mexican restaurants."""# Fill in your answer here
try:
    preferences = extract_preferences(test_message, llm) # Fill in your answer here
    print("Extracted Preferences:")
    print(json.dumps(preferences, indent=2))
except Exception as e:
    print(f"Requires valid OpenAI API key. Error: {e}")

Extracted Preferences:
{
  "favorite_cuisines": [
    "Mexican"
  ],
  "dietary_restrictions": [
    "vegetarian"
  ],
  "dining_occasion": "not specified",
  "price_range": "$",
  "flavor_preferences": [],
  "other_preferences": "affordable"
}


Integrate the multi-agent workflow
Now we'll create a simplified version of the multi-agent workflow that can be called from the chatbot.

In [12]:
def run_recommendation_workflow(preferences: Dict[str, Any], recommendation_type: str) -> Dict[str, Any]:
    """Run the multi-agent workflow and return recommendations.
    
    Args:
        preferences: User preferences extracted from their message
        recommendation_type: "restaurant", "recipe", or "both"
    
    Returns:
        Dictionary with recommendations
    """
    
    # For this demo, we'll create mock recommendations
    # In a real implementation, this would call the LangGraph workflow from Lesson 2
    
    print(f"Running workflow for {recommendation_type} recommendations...")
    
    mock_recommendations = {
        "restaurants": [
            {
                "name": "Green Leaf Bistro",
                "cuisine": "Mediterranean",
                "price": "$$",
                "reasoning": "This restaurant perfectly aligns with your preference for healthy, plant-based options and offers a diverse Mediterranean menu with excellent vegetarian choices."
            },
            {
                "name": "Spice Route",
                "cuisine": "Indian",
                "price": "$$",
                "reasoning": "Known for authentic Indian cuisine with extensive vegetarian options. The spice level can be customized to your preference."
            }
        ],
        "recipes": [
            {
                "name": "One-Pot Chickpea Curry",
                "cuisine": "Indian",
                "difficulty": "Easy",
                "reasoning": "A flavorful, protein-rich dish that matches your love for bold flavors. Ready in 30 minutes with simple ingredients."
            },
            {
                "name": "Mediterranean Quinoa Bowl",
                "cuisine": "Mediterranean",
                "difficulty": "Easy",
                "reasoning": "Nutritious and satisfying, this bowl combines your favorite Mediterranean flavors with plant-based protein."
            }
        ]
    }
    
    # Filter based on recommendation type
    if recommendation_type == "restaurant":
        return {"restaurants": mock_recommendations["restaurants"]}
    elif recommendation_type == "recipe":
        return {"recipes": mock_recommendations["recipes"]}
    else:  # both
        return mock_recommendations

print("Workflow integration function created!")

Workflow integration function created!


Format the recommendations
Create a function to format recommendations in a user-friendly way.

In [13]:
def format_recommendations(recommendations: Dict[str, Any]) -> str:
    """Format recommendations for display in the chat."""
    
    output = ""
    
    # Format restaurant recommendations
    if "restaurants" in recommendations and recommendations["restaurants"]:
        output += "🍽️ **Restaurant Recommendations:**\n\n"
        for i, restaurant in enumerate(recommendations["restaurants"], 1):
            output += f"**{i}. {restaurant['name']}**\n"
            output += f"   - Cuisine: {restaurant['cuisine']}\n"
            output += f"   - Price: {restaurant['price']}\n"
            output += f"   - Why: {restaurant['reasoning']}\n\n"
    
    # Format recipe recommendations
    if "recipes" in recommendations and recommendations["recipes"]:
        output += "👨‍🍳 **Recipe Recommendations:**\n\n"
        for i, recipe in enumerate(recommendations["recipes"], 1):
            output += f"**{i}. {recipe['name']}**\n"
            output += f"   - Cuisine: {recipe['cuisine']}\n"
            output += f"   - Difficulty: {recipe['difficulty']}\n"
            output += f"   - Why: {recipe['reasoning']}\n\n"
    
    if not output:
        output = "I couldn't generate recommendations. Please try again with more details about your preferences."
    
    return output

print("Formatting function created!")

Formatting function created!


In [15]:
def recommendation_chatbot(message: str, history: List[Tuple[str, str]]) -> str:
    """Main chatbot function that handles user requests."""
    
    try:
        # Step 1: Classify intent
        intent = classify_intent(message, llm)
        print(f"Classified intent: {intent}")
        
        # Step 2: Handle different intents
        if intent == "clarification":
            return """I'm your food recommendation assistant! I can help you with:
            
🍽️ **Restaurant recommendations** - Tell me your cuisine preferences, dietary restrictions, and occasion
👨‍🍳 **Recipe recommendations** - Let me know what you'd like to cook
📝 **Database management** - Add, update, or delete restaurants and recipes

Just describe what you're looking for, and I'll provide personalized recommendations!"""
        
        elif intent == "database":
            return """To manage the database, please use the tabs above:
            
- **Add Restaurant**: Submit a new restaurant
- **Add Recipe**: Submit a new recipe
- **Edit/Delete**: Modify or remove existing entries

Is there anything else I can help you with?"""
        
        elif intent in ["restaurant", "recipe", "both"]:
            # Step 3: Extract preferences
            preferences = extract_preferences(message, llm)
            print(f"Extracted preferences: {preferences}")
            
            # Step 4: Run workflow
            recommendations = run_recommendation_workflow(preferences, intent)
            
            # Step 5: Format output
            formatted_output = format_recommendations(recommendations)
            
            return formatted_output
        
        else:
            return "I'm not sure how to help with that. Can you rephrase your request?"
    
    except Exception as e:
        return f"I encountered an error: {str(e)}. Please make sure you have set your OpenAI API key."

print("Complete chatbot function created!")

Complete chatbot function created!


In [16]:
def add_restaurant(name: str, cuisine: str, price: str, location: str, description: str) -> str:
    """Add a new restaurant to the database."""
    # In a real implementation, this would add to the vector database
    print(f"Adding restaurant: {name}")
    return f"✅ Successfully added '{name}' to the database!"

def add_recipe(name: str, cuisine: str, difficulty: str, prep_time: str, ingredients: str, instructions: str) -> str:
    """Add a new recipe to the database."""
    # In a real implementation, this would add to the vector database
    print(f"Adding recipe: {name}")
    return f"✅ Successfully added '{name}' recipe to the database!"

print("Database management functions created!")

Database management functions created!


In [18]:
#Create the main interface with tabs
with gr.Blocks(title="Food Recommendation Chatbot", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # 🍽️ Food Recommendation Chatbot
    
    Your personal AI assistant for restaurant and recipe recommendations!
    """)
    
    with gr.Tabs():
        
        # Tab 1: Chat Interface
        with gr.Tab("💬 Chat"):
            chatbot_interface = gr.ChatInterface(
                fn=recommendation_chatbot,
                examples=[
                    "I'm looking for vegetarian restaurants",
                    "Suggest some easy recipes for dinner",
                    "I want spicy Thai food recommendations",
                    "What can you help me with?"
                ],
                title="Chat with the Recommendation Assistant",
                description="Describe your food preferences and I'll recommend restaurants or recipes!"
            )
        
        # Tab 2: Add Restaurant
        with gr.Tab("➕ Add Restaurant"):
            gr.Markdown("### Add a New Restaurant to the Database")
            
            with gr.Row():
                with gr.Column():
                    rest_name = gr.Textbox(label="Restaurant Name")
                    rest_cuisine = gr.Textbox(label="Cuisine Type")
                    rest_price = gr.Dropdown(
                        choices=["$", "$$", "$$$", "$$$$"],
                        label="Price Range"
                    )
                with gr.Column():
                    rest_location = gr.Textbox(label="Location")
                    rest_description = gr.Textbox(
                        label="Description",
                        lines=3
                    )
            
            add_rest_btn = gr.Button("Add Restaurant", variant="primary")
            rest_output = gr.Textbox(label="Status")
            
            add_rest_btn.click(
                fn=add_restaurant,
                inputs=[rest_name, rest_cuisine, rest_price, rest_location, rest_description],
                outputs=rest_output
            )
        
        # Tab 3: Add Recipe
        with gr.Tab("➕ Add Recipe"):
            gr.Markdown("### Add a New Recipe to the Database")
            
            with gr.Row():
                with gr.Column():
                    recipe_name = gr.Textbox(label="Recipe Name")
                    recipe_cuisine = gr.Textbox(label="Cuisine Type")
                    recipe_difficulty = gr.Dropdown(
                        choices=["Easy", "Medium", "Hard"],
                        label="Difficulty"
                    )
                with gr.Column():
                    recipe_time = gr.Textbox(label="Prep Time")
                    recipe_ingredients = gr.Textbox(
                        label="Ingredients (comma-separated)",
                        lines=3
                    )
            
            recipe_instructions = gr.Textbox(
                label="Instructions",
                lines=5
            )
            
            add_recipe_btn = gr.Button("Add Recipe", variant="primary")
            recipe_output = gr.Textbox(label="Status")
            
            add_recipe_btn.click(
                fn=add_recipe,
                inputs=[recipe_name, recipe_cuisine, recipe_difficulty, recipe_time, recipe_ingredients, recipe_instructions],
                outputs=recipe_output
            )
        
        # Tab 4: About
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## About This Chatbot
            
            This chatbot uses a multi-agent AI system to provide personalized food recommendations.
            
            ### Features:
            - 🤖 **Intelligent Agents**: Six specialized AI agents work together to analyze your preferences
            - 🔍 **Smart Search**: Vector database retrieval finds the most relevant options
            - 🎯 **Personalized**: Recommendations tailored to your tastes and dietary needs
            - 📝 **Editable Database**: Add your favorite restaurants and recipes
            
            ### How to Use:
            1. Go to the **Chat** tab
            2. Describe what you're looking for (cuisine, dietary restrictions, occasion, etc.)
            3. Receive personalized restaurant or recipe recommendations
            4. Use the **Add** tabs to contribute to the database
            
            ### Technologies:
            - LangChain & LangGraph for multi-agent orchestration
            - OpenAI GPT-4 for language understanding
            - Vector databases for semantic search
            - Gradio for the user interface
            """)

print("Complete interface created!")
print("\nTo launch the chatbot, run: demo.launch()")

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_52529/1120314336.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Food Recommendation Chatbot", theme=gr.themes.Soft()) as demo:


Complete interface created!

To launch the chatbot, run: demo.launch()


In [19]:
# Uncomment the line below to launch the chatbot
demo.launch(share=True)

print("To launch the chatbot interface, uncomment and run the line above.")
print("The chatbot will open in a new browser tab.")

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://21d5879b1fe927299e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


To launch the chatbot interface, uncomment and run the line above.
The chatbot will open in a new browser tab.
Classified intent: restaurant
Extracted preferences: {'favorite_cuisines': [], 'dietary_restrictions': ['vegetarian'], 'dining_occasion': 'not specified', 'price_range': 'not specified', 'flavor_preferences': [], 'other_preferences': ''}
Running workflow for restaurant recommendations...


In [20]:
# Test the chatbot with a sample message
test_message = "I'm looking for healthy vegetarian restaurants for a date night"

print("Testing chatbot with message:")
print(f"User: {test_message}\n")

try:
    response = recommendation_chatbot(test_message, [])
    print("Bot Response:")
    print(response)
except Exception as e:
    print(f"Test requires valid OpenAI API key. Error: {e}")

Testing chatbot with message:
User: I'm looking for healthy vegetarian restaurants for a date night

Classified intent: restaurant
Extracted preferences: {'favorite_cuisines': [], 'dietary_restrictions': ['vegetarian'], 'dining_occasion': 'date night', 'price_range': 'not specified', 'flavor_preferences': [], 'other_preferences': ['healthy']}
Running workflow for restaurant recommendations...
Bot Response:
🍽️ **Restaurant Recommendations:**

**1. Green Leaf Bistro**
   - Cuisine: Mediterranean
   - Price: $$
   - Why: This restaurant perfectly aligns with your preference for healthy, plant-based options and offers a diverse Mediterranean menu with excellent vegetarian choices.

**2. Spice Route**
   - Cuisine: Indian
   - Price: $$
   - Why: Known for authentic Indian cuisine with extensive vegetarian options. The spice level can be customized to your preference.


